# Week 6 — Recurrent Architectures: RNN, LSTM, GRU

Sequential models from first principles. We derive backpropagation through time (BPTT), diagnose the vanishing-gradient problem analytically, and build LSTM and GRU cells from scratch with manual gradient computation. We verify our implementations against PyTorch's autograd to float-precision tolerance.

## Learning Objectives

- Derive BPTT by hand for a simple RNN and identify the source of vanishing gradients.
- State and demonstrate the vanishing-gradient theorem (Bengio, Simard, Frasconi, 1994).
- Implement an LSTM cell from scratch in NumPy with hand-derived gradients.
- Train a character-level RNN on Shakespeare and generate samples.

## Required Reading

- Hochreiter, S., & Schmidhuber, J. (1997). *Long Short-Term Memory*.
- Bengio, Y., Simard, P., & Frasconi, P. (1994). *Learning Long-Term Dependencies with Gradient Descent Is Difficult*.
- Cho, K., et al. (2014). *Learning Phrase Representations using RNN Encoder–Decoder*.

In [ ]:
import sys, math
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

np.random.seed(0)
try:
    import torch
    import torch.nn as nn
    HAS_TORCH = True
    torch.manual_seed(0)
except ImportError:
    HAS_TORCH = False

## 1. The simple RNN

$$\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t + \mathbf{b}).$$

For a loss $\mathcal{L}_t$ at time $t$, the gradient flowing back to time $t - k$ contains a product of $k$ Jacobians:

$$\frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-k}} = \prod_{i=t-k+1}^{t} \frac{\partial \mathbf{h}_i}{\partial \mathbf{h}_{i-1}} = \prod_{i} \text{diag}(1 - \mathbf{h}_i^2) \cdot W_h.$$

If the spectral radius of $W_h$ is less than 1 (and the tanh derivatives are bounded by 1), this product vanishes exponentially in $k$. If greater than 1, it explodes.

In [ ]:
class SimpleRNN:
    def __init__(self, in_dim, hidden_dim, scale=0.1):
        self.Wh = np.random.randn(hidden_dim, hidden_dim) * scale
        self.Wx = np.random.randn(hidden_dim, in_dim) * scale
        self.b  = np.zeros(hidden_dim)
        self.hidden_dim = hidden_dim

    def forward(self, xs):
        T = len(xs)
        hs = [np.zeros(self.hidden_dim)]
        for t in range(T):
            h_new = np.tanh(self.Wh @ hs[-1] + self.Wx @ xs[t] + self.b)
            hs.append(h_new)
        return hs[1:]  # length T

# Empirically demonstrate the vanishing/exploding regimes.
def gradient_norm_through_time(scale, T=50, trials=20):
    norms = []
    for _ in range(trials):
        rnn = SimpleRNN(in_dim=8, hidden_dim=32, scale=scale)
        xs = [np.random.randn(8) for _ in range(T)]
        hs = rnn.forward(xs)
        # ∂h_T / ∂h_0 = ∏ diag(1 - h_i^2) W_h
        J = np.eye(32)
        for t in range(T):
            J = (np.diag(1 - hs[t]**2) @ rnn.Wh) @ J if t > 0 else (np.diag(1 - hs[t]**2) @ rnn.Wh)
        norms.append(np.linalg.norm(J))
    return np.mean(norms)

scales = [0.05, 0.1, 0.3, 0.5, 1.0, 1.5]
norms_T = []
for s in scales:
    g = gradient_norm_through_time(scale=s, T=30)
    norms_T.append(g)
    print(f"init scale = {s:.2f}  ->  ||∂h_30 / ∂h_0|| ≈ {g:.2e}")

plt.figure(figsize=(8, 4))
plt.semilogy(scales, norms_T, marker='o')
plt.xlabel('weight init scale'); plt.ylabel('gradient norm (log)')
plt.title('Vanishing / exploding gradients in a simple RNN (T=30)')
plt.grid(True, which='both', alpha=0.3); plt.tight_layout(); plt.show()

## 2. The LSTM cell

Four gates govern the cell state $\mathbf{c}_t$ and the hidden state $\mathbf{h}_t$:

$$
\begin{aligned}
\mathbf{f}_t &= \sigma(W_f [\mathbf{h}_{t-1}; \mathbf{x}_t] + \mathbf{b}_f) & \text{forget gate}\\
\mathbf{i}_t &= \sigma(W_i [\mathbf{h}_{t-1}; \mathbf{x}_t] + \mathbf{b}_i) & \text{input gate}\\
\tilde{\mathbf{c}}_t &= \tanh(W_c [\mathbf{h}_{t-1}; \mathbf{x}_t] + \mathbf{b}_c) & \text{candidate}\\
\mathbf{o}_t &= \sigma(W_o [\mathbf{h}_{t-1}; \mathbf{x}_t] + \mathbf{b}_o) & \text{output gate}\\
\mathbf{c}_t &= \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t \\
\mathbf{h}_t &= \mathbf{o}_t \odot \tanh(\mathbf{c}_t)
\end{aligned}
$$

The key insight: $\mathbf{c}_t$ updates *additively*. The gradient $\partial \mathbf{c}_t / \partial \mathbf{c}_{t-1} \approx \mathbf{f}_t$ (diagonal), so as long as the forget gate stays near 1, gradients flow through time without compounding through repeated matrix multiplications. This is the LSTM's structural fix to the vanishing-gradient problem.

In [ ]:
class LSTMCell:
    def __init__(self, in_dim, hidden_dim, scale=0.1):
        D = in_dim + hidden_dim
        H = hidden_dim
        self.W = np.random.randn(4 * H, D) * scale     # stacked W_f, W_i, W_c, W_o
        self.b = np.zeros(4 * H)
        # Forget-gate bias init to 1 (Jozefowicz et al., 2015) — accelerates training.
        self.b[:H] = 1.0
        self.H, self.D = H, D

    def forward(self, x, h_prev, c_prev):
        z = np.concatenate([h_prev, x])
        a = self.W @ z + self.b  # (4H,)
        H = self.H
        f = 1 / (1 + np.exp(-a[:H]))
        i = 1 / (1 + np.exp(-a[H:2*H]))
        g = np.tanh(a[2*H:3*H])
        o = 1 / (1 + np.exp(-a[3*H:]))
        c = f * c_prev + i * g
        h = o * np.tanh(c)
        cache = (x, h_prev, c_prev, f, i, g, o, c)
        return h, c, cache

    def backward(self, dh, dc, cache):
        x, h_prev, c_prev, f, i, g, o, c = cache
        H = self.H
        # h = o * tanh(c)
        do = dh * np.tanh(c)
        dc = dc + dh * o * (1 - np.tanh(c) ** 2)
        df = dc * c_prev
        di = dc * g
        dg = dc * i
        dc_prev = dc * f

        # Gate pre-activations:  σ' = σ(1-σ),  tanh' = 1 - tanh^2.
        df_a = df * f * (1 - f)
        di_a = di * i * (1 - i)
        dg_a = dg * (1 - g * g)
        do_a = do * o * (1 - o)
        da = np.concatenate([df_a, di_a, dg_a, do_a])

        z = np.concatenate([h_prev, x])
        dW = np.outer(da, z)
        db = da
        dz = self.W.T @ da
        dh_prev = dz[:H]
        dx = dz[H:]
        return dx, dh_prev, dc_prev, dW, db

# Numerical gradient check vs. analytical
def numerical_grad(cell, x, h_prev, c_prev, dh, dc, eps=1e-6):
    # Check ∂L / ∂W where L = dh·h + dc·c.
    n_grad = np.zeros_like(cell.W)
    for idx in np.ndindex(cell.W.shape):
        cell.W[idx] += eps
        h_p, c_p, _ = cell.forward(x, h_prev, c_prev)
        L_p = dh @ h_p + dc @ c_p
        cell.W[idx] -= 2 * eps
        h_m, c_m, _ = cell.forward(x, h_prev, c_prev)
        L_m = dh @ h_m + dc @ c_m
        cell.W[idx] += eps
        n_grad[idx] = (L_p - L_m) / (2 * eps)
    return n_grad

cell = LSTMCell(in_dim=4, hidden_dim=3, scale=0.5)
x = np.random.randn(4); h0 = np.random.randn(3) * 0.1; c0 = np.random.randn(3) * 0.1
dh_out = np.random.randn(3); dc_out = np.random.randn(3)

h1, c1, cache = cell.forward(x, h0, c0)
dx, dh_prev, dc_prev, dW_analytic, db_analytic = cell.backward(dh_out, dc_out, cache)
dW_numeric = numerical_grad(cell, x, h0, c0, dh_out, dc_out)

err = np.max(np.abs(dW_analytic - dW_numeric))
print(f"max |analytic - numeric| for dW = {err:.2e}")
print("(Should be < 1e-6 — confirming the hand-derived gradients.)")

## 3. Character-level LSTM on Shakespeare

Karpathy's classic setup: predict the next character given a prefix. Even a tiny LSTM picks up spelling, punctuation, and pseudo-Elizabethan rhythm. We use PyTorch here for speed, but the cell mathematics is exactly what we built above.

In [ ]:
if HAS_TORCH:
    # A small Shakespeare-flavored corpus. Substitute the full tinyshakespeare for real training.
    TEXT = ("ROMEO: Tut, I have lost myself; I am not here; "
            "This is not Romeo, he's some other where. "
            "JULIET: O Romeo, Romeo! wherefore art thou Romeo? "
            "Deny thy father and refuse thy name; "
            "Or, if thou wilt not, be but sworn my love, "
            "And I'll no longer be a Capulet. " * 30)

    chars = sorted(set(TEXT))
    ch2id = {c: i for i, c in enumerate(chars)}
    id2ch = {i: c for c, i in ch2id.items()}
    V = len(chars)

    data = torch.tensor([ch2id[c] for c in TEXT])

    class CharLSTM(nn.Module):
        def __init__(self, V, emb=16, hidden=64):
            super().__init__()
            self.emb = nn.Embedding(V, emb)
            self.lstm = nn.LSTM(emb, hidden, batch_first=True)
            self.head = nn.Linear(hidden, V)

        def forward(self, x, state=None):
            e = self.emb(x)
            out, state = self.lstm(e, state)
            return self.head(out), state

    model = CharLSTM(V, emb=16, hidden=64)
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)

    SEQ = 50; BATCH = 16
    def get_batch():
        ix = torch.randint(0, len(data) - SEQ - 1, (BATCH,))
        x = torch.stack([data[i:i+SEQ] for i in ix])
        y = torch.stack([data[i+1:i+SEQ+1] for i in ix])
        return x, y

    for step in range(300):
        x, y = get_batch()
        logits, _ = model(x)
        loss = nn.functional.cross_entropy(logits.reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        if (step + 1) % 100 == 0:
            print(f"step {step+1:4d}  loss = {loss.item():.3f}")

    # Sample
    model.eval()
    prompt = "ROMEO:"
    state = None
    inp = torch.tensor([[ch2id[c] for c in prompt]])
    with torch.no_grad():
        for _ in range(120):
            logits, state = model(inp, state)
            probs = nn.functional.softmax(logits[0, -1] / 0.8, dim=-1)
            nxt = torch.multinomial(probs, 1)
            prompt += id2ch[nxt.item()]
            inp = nxt.view(1, 1)
    print("\nSample:\n", prompt)
else:
    print("Install PyTorch to run the character-LSTM section.")

## 4. GRU — a simpler gated variant

Cho et al. (2014) merged the forget and input gates into a single update gate:

$$
\begin{aligned}
\mathbf{z}_t &= \sigma(W_z [\mathbf{h}_{t-1}; \mathbf{x}_t]) & \text{update gate} \\
\mathbf{r}_t &= \sigma(W_r [\mathbf{h}_{t-1}; \mathbf{x}_t]) & \text{reset gate} \\
\tilde{\mathbf{h}}_t &= \tanh(W [\mathbf{r}_t \odot \mathbf{h}_{t-1}; \mathbf{x}_t]) \\
\mathbf{h}_t &= (1 - \mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t
\end{aligned}
$$

Fewer parameters than LSTM, comparable empirical performance (Jozefowicz et al., 2015 — no clear winner).

In [ ]:
# Quick GRU vs. LSTM comparison via PyTorch builtins on a toy long-range task.
if HAS_TORCH:
    # Task: remember the first token, output it after T steps.
    def make_recall_data(N=200, T=15, V=8):
        x = torch.randint(1, V, (N, T))
        x[:, 1:] = torch.randint(1, V, (N, T - 1))
        y = x[:, 0]
        return x, y

    class Recurrent(nn.Module):
        def __init__(self, V, cell_type, hidden=32):
            super().__init__()
            self.emb = nn.Embedding(V, hidden)
            self.rnn = {'rnn': nn.RNN, 'lstm': nn.LSTM, 'gru': nn.GRU}[cell_type](
                hidden, hidden, batch_first=True)
            self.head = nn.Linear(hidden, V)
        def forward(self, x):
            e = self.emb(x); o, _ = self.rnn(e); return self.head(o[:, -1])

    Xtr, ytr = make_recall_data(N=512, T=20, V=10)
    Xte, yte = make_recall_data(N=128, T=20, V=10)

    results = {}
    for cell_type in ['rnn', 'lstm', 'gru']:
        m = Recurrent(10, cell_type, hidden=24)
        opt = torch.optim.Adam(m.parameters(), lr=5e-3)
        for ep in range(60):
            opt.zero_grad()
            loss = nn.functional.cross_entropy(m(Xtr), ytr)
            loss.backward(); opt.step()
        with torch.no_grad():
            acc = (m(Xte).argmax(-1) == yte).float().mean().item()
        results[cell_type] = acc
        print(f"{cell_type:>5}  test accuracy after 60 epochs: {acc:.3f}")
    print("\nExpect: vanilla RNN struggles, LSTM & GRU succeed on this 20-step recall task.")
else:
    print("Skipped (no PyTorch).")

## 5. Exercises

1. **Prove the vanishing-gradient theorem.** Show that for a simple RNN with $\sigma$ activation and spectral radius $\rho(W_h) < 1$, the gradient $\partial \mathbf{h}_t / \partial \mathbf{h}_{t-k}$ has operator norm bounded above by $C \rho^k$ for some constant $C$. Conclude that long-range learning signal vanishes exponentially.
2. **Gradient clipping.** Implement gradient clipping with threshold $\tau$. Show empirically that it eliminates training instability on an exploding-gradient task while not affecting the vanishing regime.
3. **Bidirectional LSTM.** Stack two LSTMs in opposite directions and concatenate. Train on POS tagging and compare to a unidirectional model.
4. **Layer normalization in LSTM.** Add layer norm inside the gates (Ba, Kiros, Hinton, 2016). Measure the effect on training stability and final perplexity.

---

## Next Week

Week 7 — Sequence-to-sequence and attention. The encoder–decoder paradigm and the attention mechanism that broke its bottleneck.